<a href="https://colab.research.google.com/github/jeffheaton/app_deep_learning/blob/main/t81_558_class_07_1_model_structure.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# T81-558: Applications of Deep Neural Networks
**Module 7: PyTorch Building Blocks**  

* Instructor: [Jeff Heaton](https://sites.washu.edu/jeffheaton/), McKelvey School of Engineering, [Washington University in St. Louis](https://engineering.washu.edu/index.html)
* For more information visit the [class website](https://sites.washu.edu/jeffheaton/t81-558/).

# Module 7 Material


* **Part 7.1: Model Structure** [[Video]]() [[Notebook]](t81_558_class_07_1_model_structure.ipynb)
* Part 7.2: Learnable Layers [[Video]]() [[Notebook]](t81_558_class_07_2_learnable_layers.ipynb)
* Part 7.3: Nonlinearities (Activations) [[Video]]() [[Notebook]](t81_558_class_07_3_transfer.ipynb)
* Part 7.4: Normalization & Regularization [[Video]]() [[Notebook]](t81_558_class_07_4_normalization.ipynb)
* Part 7.5: Shape [[Video]]() [[Notebook]](t81_558_class_07_5_shapes.ipynb)



# Google CoLab Instructions

The following code checks that Google CoLab is and sets up the correct hardware settings for PyTorch.


In [ ]:
try:
    import google.colab
    COLAB = True
    print("Note: using Google CoLab")
except:
    print("Note: not using Google CoLab")
    COLAB = False

# Make use of a GPU or MPS (Apple) if one is available.  (see module 2.5)
import torch
has_mps = torch.backends.mps.is_built()
device = "mps" if has_mps else "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Note: using Google CoLab
Using device: cpu


# Part 7.1: Model Structure

Composition defines how individual components are assembled into a complete model. Modules can be arranged sequentially or organized into more complex structures using containers and custom classes. This structure determines the overall architecture and how data flows from input to output. In PyTorch, all components are unified under the `nn.Module` abstraction, which enables flexible model design.

Before exploring the specific layers and activations covered in later sections of this module, it is worth understanding how PyTorch organizes those pieces into a working model. A neural network is rarely a single operation. It is almost always a chain of transformations, each one taking the output of the previous step and producing something the next step can consume. The way you express that chain in code shapes how readable, reusable, and debuggable your network becomes.

PyTorch gives you several ways to express a model, ranging from a one-line declaration of a simple feed-forward stack to a fully custom class with branching, skip connections, and conditional logic. The right choice depends on how regular your architecture is. If data flows straight through a series of layers with no detours, a sequential container is concise and clear. If the architecture has shared sub-blocks, multiple inputs, or paths that merge back together, a custom subclass of `nn.Module` is the better fit.


## The nn.Module Abstraction

Everything in PyTorch that holds learnable parameters or participates in the forward computation is an `nn.Module`. A single linear layer is an `nn.Module`. A convolutional block built from several layers is an `nn.Module`. The full network you train is also an `nn.Module`. This uniformity is the key idea behind PyTorch's design. Because every component shares the same interface, you can nest them inside one another without writing special wiring code.

When you subclass `nn.Module`, you take on two responsibilities. First, you declare the submodules and parameters your network will use, which you typically do inside `__init__`. Second, you describe the forward computation by implementing the `forward` method. PyTorch handles everything else. It tracks the parameters of every submodule you assign as an attribute, registers them so the optimizer can find them, moves them to the correct device when you call `.to(device)`, and saves and restores them through the state dictionary.

This is also what makes the backward pass automatic. As long as your `forward` method uses tensor operations and other modules, PyTorch's autograd system records the operations and can compute gradients with respect to every parameter. You never write the backward pass yourself for standard layers.

## Sequential Composition

The simplest way to build a model is to stack layers in order, one after another. PyTorch provides `nn.Sequential` for exactly this case. You pass it a list of modules, and it applies them in the given order during the forward pass. The result is itself an `nn.Module`, so it behaves like any other component.

Sequential containers are useful when the architecture is a straight pipeline. A typical fully connected classifier consists of a linear layer, an activation, another linear layer, another activation, and a final output layer. There are no skip connections, no branches, and no conditional logic. Writing this with `nn.Sequential` keeps the code compact and matches the way you would describe the network on paper.

The trade-off is flexibility. A sequential container assumes the output of each layer feeds directly into the next, with no extra arguments and no alternate paths. If your network needs to combine two inputs, reuse an intermediate result later in the network, or apply different operations based on a flag, a sequential container alone is not enough. You can still use `nn.Sequential` for the regular sub-blocks of a larger model, but the overall structure will need a custom class.

## Custom Subclasses of nn.Module

When the architecture is more complex, the standard pattern is to write your own class that inherits from `nn.Module`. This gives you full control over the forward pass. Inside `forward`, you can call submodules in any order, store intermediate values, branch on conditions, and combine tensors however the design requires. Anything you can express in Python with tensor operations is valid.

Submodules are registered automatically when you assign them to `self` inside `__init__`. This registration is what makes the parameters discoverable by `model.parameters()`, which is what the optimizer iterates over. If you store a layer in a plain Python list instead of as an attribute, PyTorch will not see it. For collections of submodules, use `nn.ModuleList` or `nn.ModuleDict`, which behave like Python lists and dictionaries but properly register the contents.

A custom class is also where you express any architectural pattern that involves more than a straight stack. Residual connections add the input of a block to its output. Encoder-decoder networks split the forward pass into two stages with intermediate representations passed between them. Multi-head models route the same features through several parallel branches and concatenate the results. All of these patterns are natural to write inside a `forward` method, and they are awkward or impossible to express through a sequential container alone.

## Containers for Organizing Submodules

Beyond `nn.Sequential`, PyTorch offers `nn.ModuleList` and `nn.ModuleDict` for cases where you need to hold a collection of layers but apply them in a non-sequential way. A `ModuleList` is useful when you have a variable number of layers, such as a stack of transformer blocks whose depth is set by a configuration value. A `ModuleDict` is useful when different inputs require different processing branches, and you want to look up the right branch by name.

The important point is that these containers exist to register submodules correctly, not to control the forward pass. You still iterate over them yourself inside `forward`. They do not call the layers automatically the way `nn.Sequential` does. This separation of concerns gives you the flexibility of arbitrary control flow while keeping parameter tracking automatic.

## Model Structure in Action

The example below builds two versions of the same small classifier for tabular data. The first version uses `nn.Sequential` to express a straight stack of layers. The second version implements the equivalent network as a subclass of `nn.Module`. Both produce the same architecture and the same parameter count. Comparing them side by side shows where each style is appropriate and how the building blocks fit together.

In [ ]:
import torch
import torch.nn as nn

# Version 1: a sequential container.
# Concise, but limited to a straight pipeline.
sequential_model = nn.Sequential(
    nn.Linear(in_features=10, out_features=64),
    nn.ReLU(),
    nn.Linear(in_features=64, out_features=32),
    nn.ReLU(),
    nn.Linear(in_features=32, out_features=3),
)


# Version 2: an explicit subclass of nn.Module.
# More verbose, but gives full control over the forward pass.
class TabularClassifier(nn.Module):
    def __init__(self, input_dim=10, hidden_dim=64, num_classes=3):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim // 2)
        self.out = nn.Linear(hidden_dim // 2, num_classes)
        self.activation = nn.ReLU()

    def forward(self, x):
        x = self.activation(self.fc1(x))
        x = self.activation(self.fc2(x))
        x = self.out(x)
        return x


custom_model = TabularClassifier()

# Confirm both models accept the same input shape and produce the same output shape.
batch = torch.randn(4, 10)  # batch of 4 samples, 10 features each
print("Sequential output shape:", sequential_model(batch).shape)
print("Custom output shape:    ", custom_model(batch).shape)

# Both styles register their parameters, so .parameters() works the same way.
seq_params = sum(p.numel() for p in sequential_model.parameters())
cust_params = sum(p.numel() for p in custom_model.parameters())
print(f"Sequential parameter count: {seq_params}")
print(f"Custom parameter count:     {cust_params}")

# Printing a model shows its structure, which is useful for debugging.
print("\nSequential model:")
print(sequential_model)
print("\nCustom model:")
print(custom_model)

Sequential output shape: torch.Size([4, 3])
Custom output shape:     torch.Size([4, 3])
Sequential parameter count: 2883
Custom parameter count:     2883

Sequential model:
Sequential(
  (0): Linear(in_features=10, out_features=64, bias=True)
  (1): ReLU()
  (2): Linear(in_features=64, out_features=32, bias=True)
  (3): ReLU()
  (4): Linear(in_features=32, out_features=3, bias=True)
)

Custom model:
TabularClassifier(
  (fc1): Linear(in_features=10, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=32, bias=True)
  (out): Linear(in_features=32, out_features=3, bias=True)
  (activation): ReLU()
)


Both versions define the same network. The sequential form is shorter and reads top to bottom like a recipe. The custom form takes a few more lines, but it gives you a place to add anything that does not fit a straight pipeline, such as residual connections, multiple inputs, or layers whose behavior depends on a flag passed into `forward`. As your models grow beyond simple stacks, you will rely on the custom-class style more often, while still using `nn.Sequential` to organize the regular sub-blocks inside it.

# Module 7 Assignment

You can find the assignment here: [assignment 7](https://github.com/jeffheaton/app_deep_learning/blob/master/assignments/assignment_yourname_t81_558_class7.ipynb)